# F1 Data Exploration

Analyze the Kaggle F1 dataset to understand structure, relationships, and data quality.
This informs feature engineering for the ML model.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load all data
data_path = Path('../data/raw')

races = pd.read_csv(data_path / 'races.csv')
results = pd.read_csv(data_path / 'results.csv')
drivers = pd.read_csv(data_path / 'drivers.csv')
constructors = pd.read_csv(data_path / 'constructors.csv')
qualifying = pd.read_csv(data_path / 'qualifying.csv')
status = pd.read_csv(data_path / 'status.csv')
pit_stops = pd.read_csv(data_path / 'pit_stops.csv')
circuits = pd.read_csv(data_path / 'circuits.csv')

print('✅ All data loaded successfully')

## 1. Dataset Overview

In [ ]:
# Show basic stats for each dataset
datasets = {
    'races': races,
    'results': results,
    'drivers': drivers,
    'constructors': constructors,
    'qualifying': qualifying,
    'status': status,
    'pit_stops': pit_stops,
    'circuits': circuits,
}

for name, df in datasets.items():
    print(f'{name:15} | Shape: {df.shape} | Columns: {list(df.columns)}')
    print()

## 2. Races Dataset

In [ ]:
print('Races Info:')
print(races.head())
print(f'\nDate range: {races["date"].min()} to {races["date"].max()}')
print(f'Unique seasons: {races["year"].nunique()}')
print(f'Races per season avg: {len(races) / races["year"].nunique():.1f}')

## 3. Results Dataset (Target Variable)

In [ ]:
print('Results Info:')
print(results.head())
print(f'\nTotal race entries: {len(results)}')
print(f'\nPosition column (target variable):')
print(results['positionOrder'].describe())
print(f'\nMissing positions (DNF): {results["position"].isna().sum()}')
print(f'Null status entries: {results["statusId"].value_counts().head(10)}')

## 4. Drivers Dataset

In [ ]:
print('Drivers Info:')
print(drivers.head())
print(f'\nTotal drivers: {len(drivers)}')
print(f'Date of birth range: {drivers["dob"].min()} to {drivers["dob"].max()}')

## 5. Constructors Dataset

In [ ]:
print('Constructors Info:')
print(constructors.head())
print(f'\nTotal constructors: {len(constructors)}')

## 6. Qualifying Dataset

In [ ]:
print('Qualifying Info:')
print(qualifying.head())
print(f'\nTotal qualifying entries: {len(qualifying)}')
print(f'Qualifying sessions with data: {qualifying["q1"].notna().sum()}')
print(f'Sessions with Q2: {qualifying["q2"].notna().sum()}')
print(f'Sessions with Q3: {qualifying["q3"].notna().sum()}')

## 7. Data Merging Test

In [ ]:
# Test merging races + results
merged = results.merge(races[['raceId', 'year', 'round', 'circuitId', 'date']], on='raceId', how='left')
merged = merged.merge(drivers[['driverId', 'surname']], on='driverId', how='left')
merged = merged.merge(constructors[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_constructor'))
merged = merged.merge(circuits[['circuitId', 'name', 'location', 'country']], on='circuitId', how='left', suffixes=('', '_circuit'))

print('Merged data sample:')
print(merged[['year', 'round', 'surname', 'name', 'location_circuit', 'grid', 'positionOrder', 'points']].head(10))
print(f'\nMerged shape: {merged.shape}')
print(f'Missing values:\n{merged.isna().sum()[merged.isna().sum() > 0]}')

## 8. Data Quality Check

In [ ]:
# Check for issues
print('Data Quality Summary:')
print(f'\n1. DNF (Did Not Finish) rates:')
dnf_rate = results[results['statusId'] != 1].groupby(results['raceId'].map(races.set_index('raceId')['year'])).size() / results.groupby(results['raceId'].map(races.set_index('raceId')['year'])).size() * 100
print(f'   Average DNF rate per season: {dnf_rate.mean():.1f}%')

print(f'\n2. Grid positions (missing = practice starters):')
print(f'   Missing grid positions: {results["grid"].isna().sum()} ({results["grid"].isna().sum() / len(results) * 100:.2f}%)')

print(f'\n3. Position orders (races counted, DNF handled):')
print(f'   Entries with position: {results["positionOrder"].notna().sum()}')
print(f'   Entries without position: {results["positionOrder"].isna().sum()}')

print(f'\n4. Qualifying data coverage:')
qualifying_coverage = qualifying.groupby(qualifying['raceId'].map(races.set_index('raceId')['year'])).apply(lambda x: x['q1'].notna().sum() / len(x) * 100)
print(f'   Qualifying coverage by year: {qualifying_coverage.mean():.1f}% avg')

## 9. Time Series Analysis

In [ ]:
# Check data availability by season
races_by_year = races.groupby('year').size()
print('Races per season:')
print(races_by_year[races_by_year.index >= 2020])

# Filter to recent seasons (2020-2024)
recent_races = races[races['year'] >= 2020]['raceId'].tolist()
recent_results = results[results['raceId'].isin(recent_races)]

print(f'\nRecent data (2020-2024):')
print(f'  Races: {len(recent_races)}')
print(f'  Results entries: {len(recent_results)}')
print(f'  Unique drivers: {recent_results["driverId"].nunique()}')

## 10. Recommendations for Feature Engineering

In [ ]:
print("""\n📊 KEY FINDINGS:

1. DATA STRUCTURE:
   - Results indexed by raceId + driverId
   - Position order is primary target variable (1-indexed positions)
   - DNF marked via statusId (not position null)
   - Grid position sometimes null (practice starts)

2. FEATURES TO ENGINEER:
   - Driver stats: avg position, DNF rate, consistency
   - Constructor strength: avg team position
   - Track performance: driver avg at each circuit
   - Driver form: last 5 races average
   - Qualifying-race gap: qual position - race position
   - Pit stop data: counts per race (if available)

3. TRAIN/VAL/TEST SPLIT:
   - Training: 2020-2022 races
   - Validation: 2023 races
   - Testing: 2024 races
   - This ensures temporal consistency

4. DATA CLEANING:
   - Remove practice starts (null grid positions)
   - Handle DNF: option A) predict position with penalty, B) separate classification
   - For MVP: exclude DNF races or assign position ~20+

5. MISSING DATA:
   - Qualifying Q1/Q2/Q3 gaps: interpolate or use best available
   - Grid position nulls: drop (practice starts rare)
   - Pit stop data: may be incomplete, use count
""")

In [ ]:
# Save this notebook's analysis to reference
print('✅ Data exploration complete! Ready for feature engineering.')